# Task 3: Association Rule Mining (15 points)

## 3. Data Preprocessing for FP-growth
First, we filter the dataset and discretize the selected features using a 3:4:3 ratio.

In [5]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

# 1. Load and filter data (price_range = 1)
df = pd.read_csv('mobile_price.csv')
df_filtered = df[df['price_range'] == 1].copy()

# 2. Select specific features
features = ['ram', 'int_memory', 'px_width', 'battery_power']
df_arm = df_filtered[features].copy()

# 3. Categorization logic based on RANGE (max - min) as per PDF
def categorize_range_343(series):
    f_min = series.min()
    f_max = series.max()
    f_range = f_max - f_min
    
    # Intervals based on 3:4:3 ratio of the range
    low_thresh = f_min + 0.3 * f_range
    high_thresh = f_min + 0.7 * f_range # (0.3 + 0.4)
    
    def map_val(val):
        if val <= low_thresh: return 'low'
        if val <= high_thresh: return 'medium'
        return 'high'
    
    return series.apply(map_val)

# Apply range-based categorization and add feature prefixes
for col in features:
    df_arm[col] = categorize_range_343(df_arm[col])
    df_arm[col] = col + '_' + df_arm[col]

# Convert to transactions
transaction_list = df_arm.values.tolist()

print("Example Transaction (Range-based):")
print(transaction_list[0])

Example Transaction (Range-based):
['ram_high', 'int_memory_low', 'px_width_low', 'battery_power_low']


## 3(a) Frequent Patterns (Support $\ge$ 0.3)
Use FP-growth to find all frequent itemsets.

In [6]:
# 1. Transaction Encoding
te = TransactionEncoder()
te_ary = te.fit(transaction_list).transform(transaction_list)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

# 2. FP-growth
frequent_itemsets = fpgrowth(df_encoded, min_support=0.3, use_colnames=True)

print(f"Found {len(frequent_itemsets)} frequent itemsets with support >= 0.3")
display(frequent_itemsets.sort_values(by='support', ascending=False))

Found 8 frequent itemsets with support >= 0.3


,support,itemsets
2,0.682,(ram_medium)
3,0.416,(px_width_medium)
5,0.414,(battery_power_medium)
4,0.412,(int_memory_medium)
7,0.318,"(ram_medium, battery_power_medium)"
0,0.316,(int_memory_low)
1,0.308,(battery_power_low)
6,0.306,"(ram_medium, px_width_medium)"


## 3(b) Association Rules (Support $\ge$ 0.3, Confidence $\ge$ 0.4, Lift $\ge$ 0.8)

In [7]:
# Generate Rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.4)

# Filter by Lift
final_rules = rules[rules['lift'] >= 0.8]

print(f"Found {len(final_rules)} association rules meeting the criteria.")
display(final_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

Found 4 association rules meeting the criteria.


,antecedents,consequents,support,confidence,lift
0,(ram_medium),(px_width_medium),0.306,0.448680,1.078559
1,(px_width_medium),(ram_medium),0.306,0.735577,1.078559
2,(battery_power_medium),(ram_medium),0.318,0.768116,1.126270
3,(ram_medium),(battery_power_medium),0.318,0.466276,1.126270


## 3(c) Observations
Describe your findings from the frequent patterns and association rules above.